## Tutorial

# Calculate and Compare Canopy Water Content

## Second of two notebooks

### Authors: Bridget Hass, Hannah Rieder and Randi Neff

In this tutorial, we will learn how to evaluate forest health using a calculation of the Canopy Water Content (CWC) from individual tiles at the Soaproot Saddle (SOAP) field site in the Sierra National Forest in California. The hyperspectral data for the CWC calculation comes from the National Ecological Observatory Network's (NEON) Level 3 Spectrometer orthorectified surface directional reflectance - mosaic data product and the Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product.

## The objectives of this tutorial (divided between two notebooks) are to:

* Use co-located data from NEON and EMIT
* Calculate Canopy Water Content (CWC) from NEON and EMIT hyperspectral data
* Evaluate CWC data at different scales
* Compare between burned and unburned areas

DATA The data provided with this tutorial were derived from existing code at:

* NEON Spectrometer orthorectified surface bidirectional reflectance data.
* Shapefiles for Creek fire boundary and NEON burned and unburned tiles which are found in the DATA folder.
* EMIT L2A Estimated Surface Reflectance granule(s) that cover the NEON burned and unburned tiles.
* Land Processes Distributed Active Archive Center (LP DAAC).

Additional data will be downloaded programmatically within this tutorial.

## What we should have after completing notebook 1:

* two EMIT cropped datasets (one for the burned tile and one for the unburned tile) exported to netcdf files
* two NEON reflectance datasets (one for the burned tile and one for the unburned tile). These datasets could already have been converted from hdf5 format into xarray, have the scale factor applied, have bad bands set to NaN, have necessary data types turned from float64 to float32, and be exported to netcdf files
## Tutorial Outline for Notebook 2 - Canopy Water Content Comparison

* Extract hyperspectral reflectance data
* Calculate Canopy Water Content (CWC)
* Analyze results

In [1]:
# Import Packages
import os, sys #python module to create and acces file paths
# Some cells may generate warnings that we can ignore.
# Comment below lines to see.
import warnings
warnings.filterwarnings('ignore')

import numpy as np #work with multi-dimensional arrays
import xarray as xr #work with labelled multi-dimenstional arrays
from osgeo import gdal #work with raster and vector geospatial data
import rasterio as rio #work with geospatial raster data
import rioxarray as rxr #work with raster arrays
from matplotlib import pyplot as plt #plotting data
import hvplot.xarray #plot multi-dimensional arrays
import hvplot.pandas #plot DataFrames/Series
import pandas as pd #work with DataFrames
import geopandas as gpd #work with geospatial shapefiles
import earthaccess #search for, download, & stream NASA earth data

from modules.emit_tools import emit_xarray #open EMIT datasets into xarray.Dataset
from modules.ewt_calc import calc_ewt #calculate canopy water content fxn
from scipy.optimize import least_squares #nonlinear least-squares

from modules.test_functions import data_download_tracker, surfrfl_hvplot_image
import neonutilities as nu #work with NEON reflectance data
import h5py #work with NEON reflectance data

### Open and Process NEON Reflectance Data

### Open Cropped EMIT Data

In [ ]:
# Define filepaths
emit_burn_fp = ("../../../data"
                "/SOAP"
                "/EMIT"
                "/L2Arefl/"
                "EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burn.nc")
# emit_unburn_fp = ("../../../data"
#                   "/SOAP"
#                   "/EMIT"
#                   "/L2Arefl/"
#                   "EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burn.nc")

In [ ]:
# Open NetCDF EMIT burned and unburned filepaths
emit_burn_ds = xr.open_dataset(emit_burn_fp, decode_coords="all")

# emit_unburn_ds = xr.open_dataset(emit_unburn_fp, decode_coords="all")


In [ ]:
# Check datasets
emit_burn_ds

In [ ]:
# Check datasets
#emit_unburn_ds

### Calculate Canopy Water Content (CWC)

#### Define necessary functions and files

ADD NOTES ABOUT WHAT BEER LAMBERT FXN IS AND WHAT THE K_LIQUID_WATER_ICE.CSV IS AND WHERE TO GET THE K_LIQUID... FILE.

In [ ]:
# https://github.com/isofit/isofit/blob/main/isofit/inversion/inverse_simple.py#L514C1-L532C17
def beer_lambert_model(x, y, wl, alpha_lw):
    """Function, which computes the vector
    of residuals between measured and modeled
    surface reflectance optimizing for path
    length of surface liquid water based on
    the Beer-Lambert attenuation law.

    Args:
        x: state vector (liquid water path length, intercept, slope)
        y: measurement (surface reflectance spectrum)
        wl: instrument wavelengths
        alpha_lw: wavelength dependent absorption coefficients of liquid water

    Returns:
        resid: residual between modeled and measured surface reflectance
    """

    attenuation = np.exp(-x[0] * 1e7 * alpha_lw)
    rho = (x[1] + x[2] * wl) * attenuation
    resid = rho - y

    return resid

In [ ]:
#define file path to k_liquid_water_ice.csv file
#this .csv was originally here: C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\data\SOAP\EMIT
#moved it to this filepath: C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\notebooks\exploratory\data
#after I got an error from the calc_ewt fxn that it couldn't find the .csv file.
wp_fp = ("../data/k_liquid_water_ice.csv")

# Read k_liquid_water_ice.csv file into a DataFrame
k_wi = pd.read_csv(wp_fp)

# Check k_wi DataFrame
k_wi.head()

The function below gets the desired data from the csv file

In [ ]:
# https://github.com/isofit/isofit/blob/dev/isofit/core/common.py#L461C1-L488C26
def get_refractive_index(k_wi, a, b, col_wvl, col_k):
    """Convert refractive index table entries to numpy array.

    Args:
        k_wi:    variable
        a:       start line
        b:       end line
        col_wvl: wavelength column in pandas table
        col_k:   k column in pandas table

    Returns:
        wvl_arr: array of wavelengths
        k_arr:   array of imaginary parts of refractive index
    """

    wvl_ = []
    k_ = []

    for ii in range(a, b):
        wvl = k_wi.at[ii, col_wvl]
        k = k_wi.at[ii, col_k]
        wvl_.append(wvl)
        k_.append(k)

    wvl_arr = np.asarray(wvl_)
    k_arr = np.asarray(k_)

    return wvl_arr, k_arr

The fxn below "uses least squares optimization to minimize the residuals of our Beer-Lambert Model and find a likely path length of liquid water" (from existing NASA EMIT CWC notebook).

In [ ]:
# https://github.com/isofit/isofit/blob/main/isofit/inversion/inverse_simple.py#L443C1-L511C24
def invert_liquid_water(
    rfl_meas: np.array,
    wl: np.array,
    l_shoulder: float = 850,
    r_shoulder: float = 1100,
    lw_init: tuple = (0.02, 0.3, 0.0002),
    lw_bounds: tuple = ([0, 0.5], [0, 1.0], [-0.0004, 0.0004]),
    ewt_detection_limit: float = 0.5,
    return_abs_co: bool = False,
):
    """Given a reflectance estimate, fit a state vector including liquid water path length
    based on a simple Beer-Lambert surface model.

    Args:
        rfl_meas:            surface reflectance spectrum
        wl:                  instrument wavelengths, must be same size as rfl_meas
        l_shoulder:          wavelength of left absorption feature shoulder
        r_shoulder:          wavelength of right absorption feature shoulder
        lw_init:             initial guess for liquid water path length, intercept, and slope
        lw_bounds:           lower and upper bounds for liquid water path length, intercept, and slope
        ewt_detection_limit: upper detection limit for ewt
        return_abs_co:       if True, returns absorption coefficients of liquid water

    Returns:
        solution: estimated liquid water path length, intercept, and slope based on a given surface reflectance
    """
    
    # Ensure least squares is done with float64 datatype (added)
    wl = np.float64(wl)
    
    # params needed for liquid water fitting
    lw_feature_left = np.argmin(abs(l_shoulder - wl))
    lw_feature_right = np.argmin(abs(r_shoulder - wl))
    wl_sel = wl[lw_feature_left : lw_feature_right + 1]

    # adjust upper detection limit for ewt if specified
    if ewt_detection_limit != 0.5:
        lw_bounds[0][1] = ewt_detection_limit

    # load imaginary part of liquid water refractive index and calculate wavelength dependent absorption coefficient
    # __file__ should live at isofit/isofit/inversion/
    
    
    data_dir_path = "../data"
    path_k = os.path.join(data_dir_path,"k_liquid_water_ice.csv")
    
    #isofit_path = os.path.dirname(os.path.dirname(os.path.dirname(__file__)))
    #path_k = os.path.join(isofit_path, "data", "iop", "k_liquid_water_ice.xlsx")

    # k_wi = pd.read_excel(io=path_k, sheet_name="Sheet1", engine="openpyxl")
    # wl_water, k_water = get_refractive_index(
    #     k_wi=k_wi, a=0, b=982, col_wvl="wvl_6", col_k="T = 20°C"
    # )
    k_wi = pd.read_csv(path_k)
    wl_water, k_water = get_refractive_index(
        k_wi=k_wi, a=0, b=982, col_wvl="wvl_6", col_k="T = 20°C"
    )
    kw = np.interp(x=wl_sel, xp=wl_water, fp=k_water)
    abs_co_w = 4 * np.pi * kw / wl_sel

    rfl_meas_sel = rfl_meas[lw_feature_left : lw_feature_right + 1]

    x_opt = least_squares(
        fun=beer_lambert_model,
        x0=lw_init,
        jac="2-point",
        method="trf",
        bounds=(
            np.array([lw_bounds[ii][0] for ii in range(3)]),
            np.array([lw_bounds[ii][1] for ii in range(3)]),
        ),
        max_nfev=15,
        args=(rfl_meas_sel, wl_sel, abs_co_w),
    )

    solution = x_opt.x

    if return_abs_co:
        return solution, abs_co_w
    else:
        return solution

#### Calculate CWC using the calc_ewt function imported in the beginning

In [ ]:
# Learn about calc_ewt function
help(calc_ewt)

In [ ]:
# Set output directory where results will be stored
emit_out_dir = "../../../data/SOAP/EMIT/CWC/"
#add neon output directory here too? maybe just go back to one output directory?

In [ ]:
# Start to a possible conditional statement for CWC calculation
cwc_soap_burn_fp = r'C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\data\SOAP\EMIT\CWC\EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burned_CWC.nc'
#if cwc has already been calculated and saved to a netcdf file,
if os.path.exists(cwc_soap_burn_fp):
    print('path exists')
    #open the CWC netcdf file and
    cwc_ds = xr.open_dataset(cwc_soap_burn_fp, decode_coords="all")
    #cwc_ds = emit_xarray(cwc_filepath)
    #display the CWC dataset
    display(cwc_ds)
else:
    print('path does not exist, calculating CWC...')
    #commented the %%time out b/c it threw an error, CWC still calculated w/o it for the burned tile
    #%%time
    emit_burn_cwc_ds = calc_ewt(
        #burned emit dataset
        emit_burn_fp,
        out_dir,
        ewt_detection_limit=1.5,
        return_cwc=True
    )
    display(emit_burn_cwc_ds)

# The above code all works, it just needs to be changes so it is reproducible and not specific to my file paths!

Below is non-conditional code to calculate CWC for emit_burn_ds. If the %%time isn't the first thing in the cell, it doesn't work. Need to investigate the %%time code and see how necessary it is.

In [ ]:
# %%time
# emit_burn_cwc_ds = calc_ewt(
#     #burned emit dataset
#     emit_burn_fp,
#     emit_out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

# # View emit_burn_cwc_ds 
# emit_burn_cwc_ds

### Visualize CWC

In [ ]:
# Plot CWC of the SOAP burned tile using surfrfl_hvplot_image fxn
surfrfl_hvplot_image(
    emit_burn_cwc_ds,
    plottitle=f"SOAP Burned Tile {emit_burn_cwc_ds.cwc.long_name} ({emit_burn_cwc_ds.cwc.units}) July 31, 2023",
clabel="Canopy Water Content (g/cm^2)")

In [ ]:
#export CWC datasets to NetCDF files so we don't have to run the CWC calculations again
# emit_burn_cwc_ds.to_netcdf("../../../data"
#            "/SOAP"
#            "/EMIT"
#            "/CWC/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burned_CWC.nc")

### Compare CWC Calculations